# Experiment Tracking with MLflow

MLflow is the industry standard for tracking ML experiments, packaging models, and managing the ML lifecycle.

1. **MLflow Tracking** - Log parameters, metrics, and artifacts
2. **Comparing Runs** - Find the best model configuration
3. **Model Logging** - Save and load models with MLflow
4. **Autologging** - Automatic tracking for sklearn, PyTorch, etc.

**Setup**: `pip install mlflow` then run `mlflow ui` in terminal to view at http://localhost:5000

In [ ]:
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import matplotlib.pyplot as plt

In [ ]:
# Setup
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Set experiment name
mlflow.set_experiment("breast_cancer_classification")

## 1. Manual Logging

In [ ]:
def train_and_log(model_name, model, X_train, X_test, y_train, y_test, params=None):
    """Train model and log everything to MLflow."""
    with mlflow.start_run(run_name=model_name):
        # Log parameters
        if params:
            mlflow.log_params(params)
        mlflow.log_param("model_type", model_name)
        mlflow.log_param("train_size", len(X_train))
        
        # Train
        pipe = Pipeline([("scaler", StandardScaler()), ("model", model)])
        pipe.fit(X_train, y_train)
        
        # Evaluate
        y_pred = pipe.predict(X_test)
        y_prob = pipe.predict_proba(X_test)[:, 1]
        
        metrics = {
            "accuracy": accuracy_score(y_test, y_pred),
            "f1_score": f1_score(y_test, y_pred),
            "roc_auc": roc_auc_score(y_test, y_prob),
        }
        
        # Log metrics
        mlflow.log_metrics(metrics)
        
        # Log model
        mlflow.sklearn.log_model(pipe, "model")
        
        # Log a custom artifact (e.g., feature importance plot)
        if hasattr(model, "feature_importances_"):
            fig, ax = plt.subplots(figsize=(8, 6))
            importance = pd.Series(model.feature_importances_, index=data.feature_names)
            importance.nlargest(10).plot(kind="barh", ax=ax)
            ax.set_title(f"{model_name} Feature Importance")
            plt.tight_layout()
            fig.savefig("feature_importance.png")
            mlflow.log_artifact("feature_importance.png")
            plt.close()
        
        print(f"{model_name}: accuracy={metrics['accuracy']:.4f}, f1={metrics['f1_score']:.4f}, auc={metrics['roc_auc']:.4f}")
        return metrics

In [ ]:
# Run experiments with different models
experiments = [
    ("LogisticRegression", LogisticRegression(C=1.0, max_iter=5000), {"C": 1.0}),
    ("LogisticRegression_L1", LogisticRegression(C=0.1, penalty="l1", solver="saga", max_iter=5000), {"C": 0.1, "penalty": "l1"}),
    ("RandomForest", RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42), {"n_estimators": 100, "max_depth": 5}),
    ("RandomForest_deep", RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42), {"n_estimators": 200, "max_depth": 10}),
    ("GradientBoosting", GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42), {"n_estimators": 100, "max_depth": 3}),
]

all_results = {}
for name, model, params in experiments:
    all_results[name] = train_and_log(name, model, X_train, X_test, y_train, y_test, params)

## 2. Autologging

MLflow can automatically log parameters, metrics, and models for supported frameworks.

In [ ]:
# Enable autologging for sklearn
mlflow.sklearn.autolog()

with mlflow.start_run(run_name="autolog_rf"):
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", RandomForestClassifier(n_estimators=150, max_depth=7, random_state=42)),
    ])
    pipe.fit(X_train, y_train)
    # Everything is logged automatically!

mlflow.sklearn.autolog(disable=True)  # Disable after use
print("Autologging complete - check MLflow UI for details")

## 3. Loading a Model from MLflow

In [ ]:
# Search for the best run
experiment = mlflow.get_experiment_by_name("breast_cancer_classification")
if experiment:
    runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=["metrics.roc_auc DESC"],
        max_results=5,
    )
    print("Top 5 runs by ROC-AUC:")
    print(runs[["run_id", "params.model_type", "metrics.accuracy", "metrics.f1_score", "metrics.roc_auc"]].to_string())
    
    # Load best model
    # best_run_id = runs.iloc[0]["run_id"]
    # loaded_model = mlflow.sklearn.load_model(f"runs:/{best_run_id}/model")
    # y_pred_loaded = loaded_model.predict(X_test)
    # print(f"\nLoaded model accuracy: {accuracy_score(y_test, y_pred_loaded):.4f}")

## Key Takeaways

1. **Track every experiment** - you'll thank yourself when you need to reproduce results
2. **Log parameters, metrics, AND artifacts** (plots, data profiles, model files)
3. **Use autologging** for quick wins, manual logging for custom metrics
4. **MLflow UI** is invaluable for comparing runs side-by-side
5. **Model logging enables reproducibility** - load any previous model by run ID
6. **In production**: point MLflow to a shared tracking server (PostgreSQL + S3)